# Fraud Detection Experiments - Neo4j Graph Features on Vertex AI

## Before you run

**Step 1 - Prepare data (run once on your local machine):**
```bash
# Creates artifacts/demo_dataset.parquet and uploads to GCS
python demo/prepare_demo_data.py --bucket your-gcs-bucket
```

**Step 2 - Set DATASET_PATH below** to your GCS path (or local path for testing).

**Step 3 - Run all cells** (Runtime > Run all).

---

### What the experiments compare

| | Experiment 1 | Experiment 2 |
|---|---|---|
| **Name** | Tabular Baseline | Graph-Enhanced |
| **Features** | Raw transaction columns only | Tabular + graph features + FastRP embeddings |
| **Graph context** | None | Entity fraud rates, temporal card chain, 64-dim node embeddings |

Both experiments use the same LightGBM architecture and hyperparameters. The only difference is the feature set.

In [ ]:
# Install LightGBM if not already available on this Workbench instance
import importlib
if importlib.util.find_spec('lightgbm') is None:
    import subprocess
    subprocess.run(['pip', 'install', 'lightgbm', '--quiet'], check=True)
    print('lightgbm installed.')
else:
    print('lightgbm already available.')

In [ ]:
# ── Data path ─────────────────────────────────────────────────────────────────
# Vertex AI (GCS):  'gs://neo4j-fraud-detection/artifacts/demo_dataset.parquet'
# Local testing:    '../artifacts/demo_dataset.parquet'
DATASET_PATH = 'gs://neo4j-fraud-detection/artifacts/demo_dataset.parquet'

# ── Results path (GCS only, optional) ─────────────────────────────────────────
RESULTS_GCS = None   # e.g. 'gs://neo4j-fraud-detection/results/'

# ── Model hyperparameters ─────────────────────────────────────────────────────
LGBM_PARAMS = dict(
    n_estimators  = 500,
    learning_rate = 0.05,
    num_leaves    = 63,
    random_state  = 42,
    n_jobs        = -1,
    verbose       = -1,
)
EARLY_STOP  = 50    # early stopping rounds
SPLIT_DT    = 12_528_000   # temporal train/val boundary (TransactionDT)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image  as mpimg
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.decomposition import PCA
import lightgbm as lgb
from matplotlib.patches import Patch

print('Imports OK.')

---
## 1. Load Dataset

`demo_dataset.parquet` is a single merged file containing:
- 590,540 transaction rows
- Raw tabular features (394 cols)
- 22 graph features extracted from Neo4j (entity fraud rates, proxy flag, temporal card chain)
- 64 FastRP node embedding dimensions computed by Neo4j GDS

In [ ]:
print(f'Loading: {DATASET_PATH}')
df = pd.read_parquet(DATASET_PATH)
print(f'Shape  : {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Fraud  : {df["isFraud"].sum():,} ({df["isFraud"].mean():.2%} of transactions)')
df[['TransactionID', 'TransactionAmt', 'isFraud', 'card_fraud_rate', 'prev_card_is_fraud', 'emb_0', 'emb_1']].head(4)

---
## 2. Feature Engineering and Train/Val Split

In [ ]:
# Label-encode categorical columns for LightGBM
CAT_COLS = ['card4', 'card6', 'ProductCD', 'P_emaildomain', 'R_emaildomain']
for col in CAT_COLS:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str).fillna('nan'))

# ── Define feature groups ──────────────────────────────────────────────────────
GRAPH_COLS = [
    'card_tx_count',         'card_fraud_count',         'card_fraud_rate',
    'payer_email_tx_count',  'payer_email_fraud_count',  'payer_email_fraud_rate',
    'recip_email_tx_count',  'recip_email_fraud_count',  'recip_email_fraud_rate',
    'billing_tx_count',      'billing_fraud_count',      'billing_fraud_rate',
    'device_tx_count',       'device_fraud_count',       'device_fraud_rate',
    'os_browser_tx_count',   'os_browser_fraud_count',   'os_browser_fraud_rate',
    'is_proxy', 'proxy_fraud_rate', 'prev_card_is_fraud', 'prev_card_dt_gap',
]
GRAPH_COLS = [c for c in GRAPH_COLS if c in df.columns]

EMB_COLS  = [f'emb_{i}' for i in range(64)]
EMB_COLS  = [c for c in EMB_COLS if c in df.columns]

EXCLUDE   = {'TransactionID', 'TransactionDT', 'isFraud'}
TAB_COLS  = [c for c in df.columns
             if c not in EXCLUDE and c not in GRAPH_COLS and c not in EMB_COLS]
ALL_COLS  = TAB_COLS + GRAPH_COLS + EMB_COLS

print(f'Tabular features   : {len(TAB_COLS)}')
print(f'Graph features     : {len(GRAPH_COLS)}')
print(f'Embedding dims     : {len(EMB_COLS)}')
print(f'Total (enhanced)   : {len(ALL_COLS)}')

# ── Temporal split ─────────────────────────────────────────────────────────────
train = df[df['TransactionDT'] <= SPLIT_DT].copy()
val   = df[df['TransactionDT'] >  SPLIT_DT].copy()

SCALE_POS = (train['isFraud'] == 0).sum() / (train['isFraud'] == 1).sum()

print(f'\nTrain : {len(train):>7,} rows  |  fraud: {train["isFraud"].mean():.2%}')
print(f'Val   : {len(val):>7,} rows  |  fraud: {val["isFraud"].mean():.2%}')
print(f'Class imbalance weight (scale_pos_weight): {SCALE_POS:.1f}')

---
## Experiment 1 - Tabular Baseline

LightGBM trained on raw tabular features only. Each transaction is scored in isolation - no knowledge of what other transactions share the same card, device, or email domain.

In [ ]:
print('Training tabular baseline ...')
print(f'  Features : {len(TAB_COLS)} tabular columns')
print(f'  Train    : {len(train):,} rows')
print()

X_tr  = train[TAB_COLS];  y_tr  = train['isFraud']
X_val = val[TAB_COLS];    y_val = val['isFraud']

params_base = {**LGBM_PARAMS, 'scale_pos_weight': SCALE_POS}
clf_base = lgb.LGBMClassifier(**params_base)
clf_base.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(EARLY_STOP, verbose=False),
        lgb.log_evaluation(50),
    ],
)
print(f'\nBest iteration: {clf_base.best_iteration_}')

In [ ]:
prob_base = clf_base.predict_proba(X_val)[:, 1]

roc_base = roc_auc_score(y_val, prob_base)
pr_base  = average_precision_score(y_val, prob_base)

# F1-optimal decision threshold
best_f1, best_t = 0, 0.5
for t in np.arange(0.05, 0.95, 0.01):
    s = f1_score(y_val, prob_base >= t, zero_division=0)
    if s > best_f1:
        best_f1, best_t = s, t
thresh_base = best_t
pred_base   = (prob_base >= thresh_base).astype(int)

f1_base   = f1_score(y_val,   pred_base)
prec_base = precision_score(y_val, pred_base, zero_division=0)
rec_base  = recall_score(y_val,  pred_base)

print('-- Experiment 1: Tabular Baseline --')
print(f'ROC-AUC   : {roc_base:.4f}')
print(f'PR-AUC    : {pr_base:.4f}  <-- primary metric')
print(f'F1        : {f1_base:.4f}   (threshold={thresh_base:.2f})')
print(f'Precision : {prec_base:.4f}')
print(f'Recall    : {rec_base:.4f}')

---
## Experiment 2 - Graph-Enhanced Model

Same LightGBM architecture. Feature set extended with:
- **22 graph features** from Neo4j: entity-level fraud rates per card, device, billing address, email domain, OS/browser; proxy flag; `prev_card_is_fraud` temporal chain signal
- **64 FastRP embedding dimensions** from Neo4j GDS: each transaction's structural fingerprint in the fraud network

In [ ]:
print('Training graph-enhanced model ...')
print(f'  Features : {len(TAB_COLS)} tabular + {len(GRAPH_COLS)} graph + {len(EMB_COLS)} embedding = {len(ALL_COLS)} total')
print(f'  Train    : {len(train):,} rows')
print()

X_tr_g  = train[ALL_COLS].fillna(-1);  y_tr_g  = train['isFraud']
X_val_g = val[ALL_COLS].fillna(-1);    y_val_g = val['isFraud']

params_graph = {**LGBM_PARAMS, 'scale_pos_weight': SCALE_POS}
clf_graph = lgb.LGBMClassifier(**params_graph)
clf_graph.fit(
    X_tr_g, y_tr_g,
    eval_set=[(X_val_g, y_val_g)],
    callbacks=[
        lgb.early_stopping(EARLY_STOP, verbose=False),
        lgb.log_evaluation(50),
    ],
)
print(f'\nBest iteration: {clf_graph.best_iteration_}')

In [ ]:
prob_graph = clf_graph.predict_proba(X_val_g)[:, 1]

roc_graph = roc_auc_score(y_val_g, prob_graph)
pr_graph  = average_precision_score(y_val_g, prob_graph)

best_f1, best_t = 0, 0.5
for t in np.arange(0.05, 0.95, 0.01):
    s = f1_score(y_val_g, prob_graph >= t, zero_division=0)
    if s > best_f1:
        best_f1, best_t = s, t
thresh_graph = best_t
pred_graph   = (prob_graph >= thresh_graph).astype(int)

f1_graph   = f1_score(y_val_g,   pred_graph)
prec_graph = precision_score(y_val_g, pred_graph, zero_division=0)
rec_graph  = recall_score(y_val_g,  pred_graph)

print('-- Experiment 2: Graph-Enhanced --')
print(f'ROC-AUC   : {roc_graph:.4f}')
print(f'PR-AUC    : {pr_graph:.4f}  <-- primary metric')
print(f'F1        : {f1_graph:.4f}   (threshold={thresh_graph:.2f})')
print(f'Precision : {prec_graph:.4f}')
print(f'Recall    : {rec_graph:.4f}')

---
## Results

In [ ]:
metrics      = ['ROC-AUC', 'PR-AUC', 'F1', 'Precision', 'Recall']
base_scores  = [roc_base,  pr_base,  f1_base,  prec_base,  rec_base]
graph_scores = [roc_graph, pr_graph, f1_graph, prec_graph, rec_graph]

results = pd.DataFrame({
    'Metric':           metrics,
    'Tabular Baseline': base_scores,
    'Graph-Enhanced':   graph_scores,
})
results['Delta'] = results['Graph-Enhanced'] - results['Tabular Baseline']
results['Lift']  = (results['Delta'] / results['Tabular Baseline'] * 100).map('{:+.1f}%'.format)

print(results.round(4).to_string(index=False))
print()
fraud_base  = int(pred_base[y_val == 1].sum())
fraud_graph = int(pred_graph[y_val_g == 1].sum())
total_fraud = int(y_val.sum())
print(f'Fraud transactions caught (val set = {total_fraud:,} total):')
print(f'  Baseline       : {fraud_base:,}  ({fraud_base/total_fraud:.1%})')
print(f'  Graph-Enhanced : {fraud_graph:,}  ({fraud_graph/total_fraud:.1%})')
print(f'  Extra caught   : +{fraud_graph - fraud_base}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Fraud Detection Experiments: Tabular Baseline vs Graph-Enhanced', fontsize=13)

# ── All metrics bar chart ──────────────────────────────────────────────────────
x = np.arange(len(metrics)); w = 0.35
b1 = axes[0].bar(x - w/2, base_scores,  w, label='Tabular Baseline', color='#3498db')
b2 = axes[0].bar(x + w/2, graph_scores, w, label='Graph-Enhanced',   color='#e74c3c')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics, rotation=20, ha='right')
axes[0].set_ylim(0.5, 1.05)
axes[0].legend(fontsize=9)
axes[0].set_title('All Metrics')
axes[0].bar_label(b1, fmt='%.3f', fontsize=8, padding=2)
axes[0].bar_label(b2, fmt='%.3f', fontsize=8, padding=2)

# ── PR-AUC focus ───────────────────────────────────────────────────────────────
axes[1].bar(
    ['Tabular\nBaseline', 'Graph-\nEnhanced'],
    [pr_base, pr_graph],
    color=['#3498db', '#e74c3c'], width=0.4, edgecolor='white'
)
axes[1].set_ylim(0, 1.0)
axes[1].set_title('PR-AUC  (primary metric - fraud is rare at 3.5%)')
for i, v in enumerate([pr_base, pr_graph]):
    axes[1].text(i, v + 0.012, f'{v:.4f}', ha='center', fontsize=13, fontweight='bold')
pct = (pr_graph - pr_base) / pr_base * 100
axes[1].set_xlabel(f'Graph improvement: {pct:+.1f}% PR-AUC', fontsize=11)

# ── Top 20 feature importances ─────────────────────────────────────────────────
feat_imp = (
    pd.DataFrame({'feature': clf_graph.feature_name_,
                  'importance': clf_graph.feature_importances_})
    .sort_values('importance', ascending=False)
    .head(20)
)

def feat_type(name):
    if name in GRAPH_COLS: return 'Graph'
    if name in EMB_COLS:   return 'Embedding'
    return 'Tabular'

color_map = {'Graph': '#e74c3c', 'Embedding': '#f39c12', 'Tabular': '#3498db'}
colors = feat_imp['feature'].map(feat_type).map(color_map).tolist()
axes[2].barh(feat_imp['feature'], feat_imp['importance'], color=colors)
axes[2].invert_yaxis()
axes[2].set_title('Top 20 Feature Importances (Graph-Enhanced)')
axes[2].set_xlabel('Importance')
axes[2].legend(
    handles=[Patch(facecolor=c, label=t) for t, c in color_map.items()],
    loc='lower right', fontsize=9
)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Confusion Matrices', fontsize=13)

ConfusionMatrixDisplay(
    confusion_matrix(y_val, pred_base),
    display_labels=['Legitimate', 'Fraud']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Experiment 1 - Tabular Baseline\nPR-AUC = {pr_base:.4f}')

ConfusionMatrixDisplay(
    confusion_matrix(y_val_g, pred_graph),
    display_labels=['Legitimate', 'Fraud']
).plot(ax=axes[1], colorbar=False, cmap='Reds')
axes[1].set_title(f'Experiment 2 - Graph-Enhanced\nPR-AUC = {pr_graph:.4f}')

plt.tight_layout()
plt.show()

In [ ]:
# PCA visualization of FastRP embeddings (64D -> 2D)
print('Projecting FastRP embeddings to 2D ...')
sample = train.sample(n=min(10_000, len(train)), random_state=42)
X_emb  = sample[EMB_COLS].fillna(0).values
y_emb  = sample['isFraud'].values

pca  = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_emb)

fig, ax = plt.subplots(figsize=(11, 6))
palette = {0: '#3498db', 1: '#e74c3c'}
for lbl, name in [(0, 'Legitimate'), (1, 'Fraud')]:
    mask = y_emb == lbl
    ax.scatter(
        X_2d[mask, 0], X_2d[mask, 1],
        c=palette[lbl], alpha=0.25 if lbl == 0 else 0.7,
        s=5 if lbl == 0 else 9,
        label=f'{name} ({mask.sum():,})'
    )

ax.set_title(
    f'FastRP Node Embeddings - PCA (2D from 64 dims, 10K sample)\n'
    f'PC1: {pca.explained_variance_ratio_[0]:.1%}  PC2: {pca.explained_variance_ratio_[1]:.1%} variance explained',
    fontsize=12,
)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(fontsize=11, markerscale=3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Save experiment results (JSON) ────────────────────────────────────────────
summary = {
    'experiment_1_tabular_baseline': {
        'roc_auc': round(roc_base,  4),
        'pr_auc':  round(pr_base,   4),
        'f1':      round(f1_base,   4),
        'precision': round(prec_base, 4),
        'recall':  round(rec_base,  4),
    },
    'experiment_2_graph_enhanced': {
        'roc_auc': round(roc_graph,  4),
        'pr_auc':  round(pr_graph,   4),
        'f1':      round(f1_graph,   4),
        'precision': round(prec_graph, 4),
        'recall':  round(rec_graph,  4),
    },
    'pr_auc_lift_pct': round((pr_graph - pr_base) / pr_base * 100, 2),
    'extra_fraud_caught': fraud_graph - fraud_base,
}
print(json.dumps(summary, indent=2))

# Optionally save to GCS
if RESULTS_GCS:
    import subprocess
    local_file = 'experiment_results.json'
    with open(local_file, 'w') as f:
        json.dump(summary, f, indent=2)
    result = subprocess.run(['gsutil', 'cp', local_file, RESULTS_GCS], capture_output=True, text=True)
    if result.returncode == 0:
        print(f'\nSaved to {RESULTS_GCS}{local_file}')
    else:
        print(f'GCS upload failed: {result.stderr.strip()}')